[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/09_ONNX_Graph_Manipulation/03_Shape_Inference/Shape_Inference_Deep_Dive.ipynb)

# 9.3 Shape Inference Algorithms — Deep Dive

## Table of Contents

| # | Section | Topic |
|---|---------|-------|
| 1 | [Type Propagation Algorithm](#section-1) | Forward dataflow, topological walk |
| 2 | [Symbolic Shape Constraints](#section-2) | `dim_param`, shape algebra, unification |
| 3 | [MatMul Shape Rules](#section-3) | Batched matrix multiply |
| 4 | [Conv Output Shapes](#section-4) | Kernel, padding, stride, dilation |
| 5 | [Reshape, Concat & Broadcast](#section-5) | Product conservation, axis summation, broadcast |
| 6 | [Dynamic vs Static Shapes](#section-6) | Concrete, symbolic, partial inference |
| 7 | [The `onnx.shape_inference` Module](#section-7) | API, strict mode, `data_prop` |
| 8 | [Interaction with Validation](#section-8) | Shape inference + checker workflow |
| 9 | [Advanced Topics](#section-9) | Custom ops, data-dependent shapes, subgraphs |

<a id="section-1"></a>
## Section 1: Type Propagation Algorithm

Shape inference is a **forward dataflow analysis** that walks the graph in topological order.
For each node, it reads input shapes and computes output shapes via operator-specific rules:

$$\text{shape}(\text{output}_i) = f_{\text{op}}\bigl(\text{shape}(\text{input}_1), \ldots, \text{shape}(\text{input}_k), \text{attrs}\bigr)$$

### Algorithm

```
┌────────────────────────────────────────────────────────────┐
│  SHAPE-INFERENCE(graph)                                    │
├────────────────────────────────────────────────────────────┤
│  1. Init type_map from graph.input & graph.initializer     │
│  2. for node in topological_sort(graph.node):              │
│       input_types  = [type_map[i] for i in node.input]     │
│       output_types = infer_fn[node.op_type](input_types,   │
│                                             node.attribute)│
│       for name, typ in zip(node.output, output_types):     │
│           type_map[name] = typ                             │
│  3. Write type_map entries into graph.value_info           │
│  Complexity: O(|V| + |E|) — single forward pass           │
└────────────────────────────────────────────────────────────┘
```

### Shape Propagation Through a Network

```
  X: float32[N, 784]       ← graph input (known)
        │
   ┌────┴─────┐
   │  MatMul   │  W1: [784,256]   →  [N,784]×[784,256] = [N,256]
   └────┬─────┘
  H1: float32[N, 256]      ← inferred
        │
   ┌────┴─────┐
   │   Relu    │              →  same shape [N, 256]
   └────┬─────┘
  R1: float32[N, 256]      ← inferred
        │
   ┌────┴─────┐
   │  MatMul   │  W2: [256,10]   →  [N,256]×[256,10] = [N,10]
   └────┬─────┘
  Y: float32[N, 10]        ← inferred
```

For graphs with **loops** (subgraphs in `Loop`, `Scan`), inference applies **fixed-point iteration**:

$$\mathcal{T}_0 \xrightarrow{\text{infer}} \mathcal{T}_1 \xrightarrow{\text{infer}} \cdots \xrightarrow{\text{infer}} \mathcal{T}_n = \mathcal{T}_{n+1}$$

Convergence is guaranteed because the type lattice has finite height and each iteration only **refines** (narrows) types.

In [ ]:
!pip install onnx onnxruntime numpy matplotlib -q

In [ ]:
import numpy as np
import onnx
from onnx import helper, TensorProto, numpy_helper, shape_inference
from onnx.checker import check_model

def get_shape(type_proto):
    """Extract human-readable shape from a TypeProto."""
    if not type_proto.tensor_type.HasField("shape"):
        return "<unknown>"
    return [d.dim_param or d.dim_value or "?" for d in type_proto.tensor_type.shape.dim]

def print_shapes(model, label=""):
    if label: print(f"\n=== {label} ===")
    for inp in model.graph.input:
        print(f"  input  {inp.name}: {get_shape(inp.type)}")
    for vi in model.graph.value_info:
        print(f"  inter  {vi.name}: {get_shape(vi.type)}")
    for out in model.graph.output:
        print(f"  output {out.name}: {get_shape(out.type)}")

print(f"ONNX version: {onnx.__version__}")

In [ ]:
# Build MLP without intermediate shapes, then infer
np.random.seed(42)
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["N", 784])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
W1 = numpy_helper.from_array(np.random.randn(784, 256).astype(np.float32) * 0.01, "W1")
W2 = numpy_helper.from_array(np.random.randn(256, 10).astype(np.float32) * 0.01, "W2")

graph = helper.make_graph(
    [helper.make_node("MatMul", ["X", "W1"], ["H"]),
     helper.make_node("Relu", ["H"], ["R"]),
     helper.make_node("MatMul", ["R", "W2"], ["Y"])],
    "mlp", [X], [Y], initializer=[W1, W2],
)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

print_shapes(model, "BEFORE shape inference")
inferred = shape_inference.infer_shapes(model)
print_shapes(inferred, "AFTER shape inference")

<a id="section-2"></a>
## Section 2: Symbolic Shape Constraints

### Symbolic Dimensions

A dimension can be **concrete** (`dim_value = 784`) or **symbolic** (`dim_param = "batch"`).
Operators impose algebraic constraints on symbolic dimensions:

- **MatMul**: $d_1 \times d_2 \;\cdot\; d_2 \times d_3 \to d_1 \times d_3$ (contraction)
- **Reshape**: $\prod_i d_i^{\text{in}} = \prod_j d_j^{\text{out}}$ (product conservation)
- **Broadcast**: $d_i' = \max(d_i^A, d_i^B)$ when one dim is 1
- **Concat(axis=k)**: $d_k^{\text{out}} = \sum_i d_k^{\text{input}_i}$

### Symbolic vs Concrete Resolution

```
Compile Time (symbolic)              Runtime (concrete)
════════════════════════             ═══════════════════
  X: ["batch", 784]                   X: [32, 784]
  W: [784, 256]                       W: [784, 256]
  H: ["batch", 256]     ──resolve──▶  H: [32, 256]
  Y: ["batch", 10]      ──resolve──▶  Y: [32, 10]
```

When two edges share the same `dim_param`, the engine treats them as the **same** variable:

$$\text{If } \text{dim\_param}(X, 0) = \text{dim\_param}(Y, 0) = \texttt{"N"} \implies \text{Add}(X, Y)_0 = \texttt{"N"}$$

In [ ]:
# Symbolic dimension propagation through a transformer-style projection
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", "seq_len", 512])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
W = numpy_helper.from_array(np.zeros((512, 128), dtype=np.float32), "W")
B = numpy_helper.from_array(np.zeros(128, dtype=np.float32), "B")

graph = helper.make_graph(
    [helper.make_node("MatMul", ["X", "W"], ["XW"]),
     helper.make_node("Add", ["XW", "B"], ["H"]),
     helper.make_node("Relu", ["H"], ["Y"])],
    "symbolic_demo", [X], [Y], initializer=[W, B],
)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
inferred = shape_inference.infer_shapes(model)

print("Symbolic shape propagation:")
for vi in list(inferred.graph.value_info) + list(inferred.graph.output):
    print(f"  {vi.name}: {get_shape(vi.type)}")
print("\n'batch' and 'seq_len' propagated unchanged; only last dim: 512 → 128.")

<a id="section-3"></a>
## Section 3: MatMul Shape Rules

The **MatMul** operator follows NumPy-style batched matrix multiplication:

$$[M, K] \otimes [K, N] \to [M, N]$$

For batched inputs, batch dimensions are broadcast:

$$\text{shape}(C) = [\text{broadcast}(\text{batch}_A, \text{batch}_B), \; M, \; N]$$

The inner dimensions must satisfy: $A_{\text{cols}} = K = B_{\text{rows}}$

| $A$ shape | $B$ shape | $C$ shape | Notes |
|-----------|-----------|-----------|-------|
| $[M, K]$ | $[K, N]$ | $[M, N]$ | Simple 2-D |
| $[B, M, K]$ | $[K, N]$ | $[B, M, N]$ | Batched × 2-D |
| $[B, M, K]$ | $[B, K, N]$ | $[B, M, N]$ | Same batch |
| $[1, M, K]$ | $[B, K, N]$ | $[B, M, N]$ | Broadcast batch |

In [ ]:
def build_matmul_model(shape_a, shape_b):
    A = helper.make_tensor_value_info("A", TensorProto.FLOAT, shape_a)
    C = helper.make_tensor_value_info("C", TensorProto.FLOAT, None)
    concrete_b = [int(d) if isinstance(d, (int, float)) else 1 for d in shape_b]
    B_init = numpy_helper.from_array(np.zeros(concrete_b, dtype=np.float32), "B")
    graph = helper.make_graph(
        [helper.make_node("MatMul", ["A", "B"], ["C"])],
        "mm", [A], [C], initializer=[B_init],
    )
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    return get_shape(shape_inference.infer_shapes(model).graph.output[0].type)

cases = [
    (["N", 784],       [784, 256],    "2-D"),
    (["B", "N", 128],  [128, 64],     "3-D × 2-D"),
    ([1, "N", 128],    [4, 128, 64],  "Batch broadcast"),
    (["B", 8, 64],     ["B", 64, 32], "Same batch"),
]
print("MatMul shape inference:")
for sa, sb, label in cases:
    print(f"  {label:16s}  A={sa}  ×  B={sb}  →  {build_matmul_model(sa, sb)}")

<a id="section-4"></a>
## Section 4: Conv Output Shapes

For 2-D convolution with input $[N, C_{in}, H_{in}, W_{in}]$, kernel $k$,
padding $p$, stride $s$, dilation $d$:

$$H_{out} = \left\lfloor \frac{H_{in} + 2p - d(k-1) - 1}{s} \right\rfloor + 1$$

### Conv Output Shape Computation Visual

```
  k=3, p=1, s=1, d=1:                  k=3, p=0, s=2, d=1:
  ┌─────────────────────┐               ┌─────────────────────┐
  │ H = ⌊(32+2-2-1)/1⌋+1│               │ H = ⌊(32+0-2-1)/2⌋+1│
  │   = ⌊31/1⌋+1 = 32   │               │   = ⌊29/2⌋+1 = 15   │
  │ → preserves spatial  │               │ → roughly halves     │
  └─────────────────────┘               └─────────────────────┘

  Pipeline:
  Input [N,3,224,224]  ─Conv(k=7,p=3,s=2)─▶  [N,64,112,112]
                       ─MaxPool(k=3,s=2,p=1)─▶ [N,64,56,56]
                       ─Conv(k=3,p=1,s=1)─▶    [N,128,56,56]
```

In [ ]:
def conv_output_shape(h, w, k, p, s, d=1):
    return (h + 2*p - d*(k-1) - 1)//s + 1, (w + 2*p - d*(k-1) - 1)//s + 1

def build_conv_model(h, w, c_in, c_out, k, p, s, d=1):
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, c_in, h, w])
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
    W_init = numpy_helper.from_array(np.zeros((c_out, c_in, k, k), dtype=np.float32), "W")
    conv = helper.make_node("Conv", ["X", "W"], ["Y"],
                            kernel_shape=[k, k], pads=[p, p, p, p],
                            strides=[s, s], dilations=[d, d])
    graph = helper.make_graph([conv], "cv", [X], [Y], initializer=[W_init])
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    dims = [x.dim_value for x in shape_inference.infer_shapes(model).graph.output[0].type.tensor_type.shape.dim]
    return dims

configs = [
    (32, 32, 3, 64, 3, 1, 1, 1),
    (32, 32, 64, 128, 3, 0, 2, 1),
    (224, 224, 3, 64, 7, 3, 2, 1),
    (56, 56, 64, 128, 3, 2, 1, 2),
]
print("Conv output shape: formula vs shape inference")
print("=" * 60)
for h, w, ci, co, k, p, s, d in configs:
    fh, fw = conv_output_shape(h, w, k, p, s, d)
    inf = build_conv_model(h, w, ci, co, k, p, s, d)
    print(f"  [{h}×{w}] k={k} p={p} s={s} d={d}  →  formula {fh}×{fw}  |  inferred {inf[2]}×{inf[3]}  ✓={fh==inf[2]}")

<a id="section-5"></a>
## Section 5: Reshape, Concat & Broadcast Rules

### Reshape: Product Conservation

$$\prod_i d_i^{\text{in}} = \prod_j d_j^{\text{out}}, \quad d_{-1} = \frac{\prod d_i^{\text{in}}}{\prod_{j \neq -1} d_j^{\text{out}}}$$

### Concat: $d_k^{\text{out}} = \sum_i d_k^{\text{input}_i}$ (all non-axis dims must match)

### Broadcasting Alignment Diagram

```
  Step 1: Align from trailing dimension, left-pad with 1s

       A: [8, 1, 6, 1]
       B: [1, 7, 1, 5]       ← padded from [7, 1, 5]

  Step 2: Per-dimension rule  d_out = max(d_A, d_B)  when one is 1

       dim 0: max(8,1)=8   dim 1: max(1,7)=7   dim 2: max(6,1)=6   dim 3: max(1,5)=5

       C: [8, 7, 6, 5]

  FAILURE: A=[3,4], B=[3,5] → dim 1: 4≠5, neither is 1 → incompatible
```

In [ ]:
# --- Reshape ---
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [2, 3, 4])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

for label, target in [("constant", [6, 4]), ("with -1", [2, -1])]:
    sv = np.array(target, dtype=np.int64)
    nodes = [
        helper.make_node("Constant", [], ["s"],
            value=helper.make_tensor("sv", TensorProto.INT64, [len(target)], sv)),
        helper.make_node("Reshape", ["X", "s"], ["Y"]),
    ]
    g = helper.make_graph(nodes, "r", [X], [Y])
    m = helper.make_model(g, opset_imports=[helper.make_opsetid("", 17)])
    dims = get_shape(shape_inference.infer_shapes(m).graph.output[0].type)
    print(f"Reshape ({label}): [2,3,4] → {target} → {dims}  (prod={2*3*4})")

# --- Concat ---
A = helper.make_tensor_value_info("A", TensorProto.FLOAT, ["N", 64])
B = helper.make_tensor_value_info("B", TensorProto.FLOAT, ["N", 128])
C = helper.make_tensor_value_info("C", TensorProto.FLOAT, ["N", 32])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
g = helper.make_graph(
    [helper.make_node("Concat", ["A", "B", "C"], ["Y"], axis=1)],
    "cat", [A, B, C], [Y],
)
m = helper.make_model(g, opset_imports=[helper.make_opsetid("", 17)])
print(f"\nConcat(axis=1): [N,64]+[N,128]+[N,32] → {get_shape(shape_inference.infer_shapes(m).graph.output[0].type)}")

# --- Broadcast ---
def build_add_model(sa, sb):
    A = helper.make_tensor_value_info("A", TensorProto.FLOAT, sa)
    B = helper.make_tensor_value_info("B", TensorProto.FLOAT, sb)
    C = helper.make_tensor_value_info("C", TensorProto.FLOAT, None)
    g = helper.make_graph([helper.make_node("Add", ["A", "B"], ["C"])], "a", [A, B], [C])
    m = helper.make_model(g, opset_imports=[helper.make_opsetid("", 17)])
    return get_shape(shape_inference.infer_shapes(m).graph.output[0].type)

print("\nBroadcast inference:")
for sa, sb, lbl in [
    (["N", 256], [256], "bias add"),
    ([8, 1, 6, 1], [7, 1, 5], "multi-dim"),
    (["B", 3, 224, 224], [1, 3, 1, 1], "channel-wise"),
]:
    print(f"  {lbl:13s}  A={sa}  +  B={sb}  →  {build_add_model(sa, sb)}")

In [ ]:
# Manual broadcast shape computation
def broadcast_shape(shape_a, shape_b):
    """Compute broadcast output shape following NumPy rules."""
    rank = max(len(shape_a), len(shape_b))
    a_pad = [1] * (rank - len(shape_a)) + list(shape_a)
    b_pad = [1] * (rank - len(shape_b)) + list(shape_b)
    result = []
    for da, db in zip(a_pad, b_pad):
        if isinstance(da, str) or isinstance(db, str):
            result.append(db if da == 1 else da if db == 1 else (da if da == db else f"?({da},{db})"))
        elif da == db: result.append(da)
        elif da == 1:  result.append(db)
        elif db == 1:  result.append(da)
        else: raise ValueError(f"Incompatible: {da} vs {db}")
    return result

print("Manual broadcast computation:")
for a, b in [([8,1,6,1], [7,1,5]), (["B",1,512], [1,"S",512]), ([3,4], [1,4])]:
    print(f"  {a} ⊕ {b} → {broadcast_shape(a, b)}")

try:
    broadcast_shape([3, 4], [3, 5])
except ValueError as e:
    print(f"  [3,4] ⊕ [3,5] → Error: {e}")

<a id="section-6"></a>
## Section 6: Dynamic vs Static Shapes

| Category | Description | Optimization |
|----------|-------------|-------------|
| **Static** | All dims concrete at graph construction | Best: full buffer pre-allocation |
| **Dynamic** | Some dims symbolic (resolved at runtime) | Good: symbolic propagation |
| **Data-dependent** | Dims depend on tensor values, not shapes | Partial: rank known, dims unknown |

```
    Fully static         Partially dynamic       Data-dependent
  ════════════════     ════════════════════     ════════════════════
  X: [4, 784]          X: ["batch", 784]       X: [4, 784]
  H: [4, 256]          H: ["batch", 256]       NonZero(X) → [2, ?]
  Y: [4, 10]           Y: ["batch", 10]        ← unknowable
```

In [ ]:
W = numpy_helper.from_array(np.zeros((784, 10), dtype=np.float32), "W")

# Case 1: Fully static
g1 = helper.make_graph(
    [helper.make_node("MatMul", ["X", "W"], ["Y"])], "s",
    [helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 784])],
    [helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)],
    initializer=[W],
)
m1 = helper.make_model(g1, opset_imports=[helper.make_opsetid("", 17)])

# Case 2: Symbolic batch
g2 = helper.make_graph(
    [helper.make_node("MatMul", ["X", "W"], ["Y"])], "d",
    [helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", 784])],
    [helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)],
    initializer=[W],
)
m2 = helper.make_model(g2, opset_imports=[helper.make_opsetid("", 17)])

# Case 3: Data-dependent (NonZero)
g3 = helper.make_graph(
    [helper.make_node("NonZero", ["X"], ["Y"])], "dd",
    [helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 5])],
    [helper.make_tensor_value_info("Y", TensorProto.INT64, None)],
)
m3 = helper.make_model(g3, opset_imports=[helper.make_opsetid("", 17)])

print("Shape inference comparison:")
for label, m in [("Static", m1), ("Symbolic", m2), ("Data-dep", m3)]:
    inf = shape_inference.infer_shapes(m)
    print(f"  {label:10s}  → Y = {get_shape(inf.graph.output[0].type)}")

# NonZero concrete example
data = np.array([[1,0,0,2],[0,0,3,0],[4,5,0,0]], dtype=np.float32)
nz = np.array(np.nonzero(data))
print(f"\nNonZero example: {data.shape} input → {nz.shape} output ({nz.shape[1]} non-zero)")
print("This count is unknowable at graph construction time.")

<a id="section-7"></a>
## Section 7: The `onnx.shape_inference` Module

### API

```python
onnx.shape_inference.infer_shapes(
    model,              # ModelProto
    check_type=False,   # also validate types
    strict_mode=False,  # raise on inference failure
    data_prop=False,    # propagate constant data for shapes
)
```

| Parameter | Effect | When to Use |
|-----------|--------|-------------|
| `check_type=True` | Validates dtype consistency | Debugging type mismatches |
| `strict_mode=True` | Raises on any inference failure | CI/CD validation |
| `data_prop=True` | Evaluates constant subgraphs to determine shapes | Reshape with computed targets |

### `data_prop` for Constant Propagation

```
  Without data_prop:                With data_prop:
  Shape([2,3,4]) → [3]             Shape([2,3,4]) → [3]
  Gather(idx=0)  → scalar          Gather(idx=0)  → 2  (evaluated!)
  Reshape target: ???               Reshape target: known → [2, 12]
```

### Limitations
- Cannot resolve shapes depending on **runtime data values** (NonZero, Where)
- Does not cross **custom op** boundaries without registered inference functions
- **Opset version sensitive**: rules may differ across opset versions

In [ ]:
# Demonstrate infer_shapes API modes
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [2, 3, 4])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
nodes = [
    helper.make_node("Constant", [], ["s"],
        value=helper.make_tensor("sv", TensorProto.INT64, [2], np.array([6, 4], dtype=np.int64))),
    helper.make_node("Reshape", ["X", "s"], ["Y"]),
]
g = helper.make_graph(nodes, "api", [X], [Y])
model = helper.make_model(g, opset_imports=[helper.make_opsetid("", 17)])

for mode, kwargs in [
    ("default",       {}),
    ("check_type",    {"check_type": True}),
    ("data_prop",     {"data_prop": True}),
    ("strict_mode",   {"strict_mode": True}),
]:
    inf = shape_inference.infer_shapes(model, **kwargs)
    print(f"  {mode:14s}  Y = {get_shape(inf.graph.output[0].type)}")

In [ ]:
# Before/after shape inference on a full CNN (LeNet-style)
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, 1, 28, 28])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
W1 = numpy_helper.from_array(np.zeros((16, 1, 5, 5), dtype=np.float32), "W1")
W2 = numpy_helper.from_array(np.zeros((32, 16, 3, 3), dtype=np.float32), "W2")
Wfc = numpy_helper.from_array(np.zeros((800, 10), dtype=np.float32), "Wfc")

nodes = [
    helper.make_node("Conv", ["X", "W1"], ["C1"], kernel_shape=[5, 5]),
    helper.make_node("Relu", ["C1"], ["R1"]),
    helper.make_node("MaxPool", ["R1"], ["P1"], kernel_shape=[2, 2], strides=[2, 2]),
    helper.make_node("Conv", ["P1", "W2"], ["C2"], kernel_shape=[3, 3]),
    helper.make_node("Relu", ["C2"], ["R2"]),
    helper.make_node("MaxPool", ["R2"], ["P2"], kernel_shape=[2, 2], strides=[2, 2]),
    helper.make_node("Constant", [], ["fs"],
        value=helper.make_tensor("sv", TensorProto.INT64, [2], np.array([-1, 800], dtype=np.int64))),
    helper.make_node("Reshape", ["P2", "fs"], ["flat"]),
    helper.make_node("MatMul", ["flat", "Wfc"], ["Y"]),
]
graph = helper.make_graph(nodes, "lenet", [X], [Y], initializer=[W1, W2, Wfc])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

print(f"BEFORE: value_info={len(model.graph.value_info)}, output Y={get_shape(model.graph.output[0].type)}")

inferred = shape_inference.infer_shapes(model)
print(f"AFTER:  value_info={len(inferred.graph.value_info)}")
for vi in inferred.graph.value_info:
    print(f"  {vi.name:5s}: {get_shape(vi.type)}")
print(f"  {'Y':5s}: {get_shape(inferred.graph.output[0].type)}  (output)")

print("\nExpected: [1,1,28,28]→Conv(k=5)→[1,16,24,24]→Pool→[1,16,12,12]")
print("  →Conv(k=3)→[1,32,10,10]→Pool→[1,32,5,5]→Reshape→[1,800]→MatMul→[1,10]")

<a id="section-8"></a>
## Section 8: Interaction with Validation

### Recommended Workflow

```
┌────────────────────────────────────────────────────────────┐
│  1. shape_inference.infer_shapes(model)                    │
│     ├─ Fills in intermediate types                         │
│     ├─ Detects shape incompatibilities                     │
│     └─ Propagates symbolic dimensions                      │
│                                                            │
│  2. onnx.checker.check_model(inferred_model)               │
│     ├─ Validates node signatures against opset spec        │
│     ├─ Checks attribute types and required attrs           │
│     └─ Verifies initializer shapes match declarations      │
│                                                            │
│  Together: dimension mismatches, broadcast failures,       │
│  missing attributes, invalid types, graph structure errors │
└────────────────────────────────────────────────────────────┘
```

In [ ]:
def validate_model(model, label):
    print(f"\n--- {label} ---")
    try:
        inferred = shape_inference.infer_shapes(model, check_type=True)
        out = get_shape(inferred.graph.output[0].type)
        print(f"  [PASS] Shape inference → output: {out}")
    except Exception as e:
        print(f"  [FAIL] Shape inference: {e}")
        inferred = model
    try:
        check_model(inferred)
        print(f"  [PASS] Model check")
    except Exception as e:
        print(f"  [FAIL] Model check: {e}")

# Valid model
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 100])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
W_ok = numpy_helper.from_array(np.zeros((100, 50), dtype=np.float32), "W")
g = helper.make_graph([helper.make_node("MatMul", ["X", "W"], ["Y"])], "ok", [X], [Y], initializer=[W_ok])
validate_model(helper.make_model(g, opset_imports=[helper.make_opsetid("", 17)]),
               "Valid: X[4,100] @ W[100,50]")

# Dimension mismatch
W_bad = numpy_helper.from_array(np.zeros((200, 50), dtype=np.float32), "W")
g2 = helper.make_graph([helper.make_node("MatMul", ["X", "W"], ["Y"])], "bad", [X], [Y], initializer=[W_bad])
validate_model(helper.make_model(g2, opset_imports=[helper.make_opsetid("", 17)]),
               "Invalid: X[4,100] @ W[200,50] — inner dim mismatch")

<a id="section-9"></a>
## Section 9: Advanced Topics

### Custom Op Shapes

Custom operators have no built-in shape rule. Solutions:
1. **Annotate `value_info`** manually before running inference
2. **Register a shape inference function** via ONNX C++ API
3. **Use `FunctionProto`** — define custom op as a subgraph of standard ops

### Data-Dependent Shapes

| Operator | Why Partial | What's Known |
|----------|------------|-------------|
| `NonZero` | Count depends on values | Rank, but not dim sizes |
| `Where` | Mask selects elements | Broadcast shape of inputs |
| `Compress` | Boolean filter | Rank only |

### Subgraph Inference

| Operator | Challenge |
|----------|-----------|
| `If` | Both branches must produce compatible output shapes |
| `Loop` | Carry variables create feedback; need fixed-point iteration |
| `Scan` | Scan outputs accumulate along a new leading axis |

In [ ]:
# Custom op: annotate value_info to enable downstream inference
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["N", 256])
Z = helper.make_tensor_value_info("Z", TensorProto.FLOAT, None)
W_out = numpy_helper.from_array(np.zeros((256, 10), dtype=np.float32), "Wout")

graph = helper.make_graph(
    [helper.make_node("MyGelu", ["X"], ["Y"], domain="my.domain"),
     helper.make_node("MatMul", ["Y", "Wout"], ["Z"])],
    "custom", [X], [Z], initializer=[W_out],
)
model = helper.make_model(graph, opset_imports=[
    helper.make_opsetid("", 17), helper.make_opsetid("my.domain", 1),
])

inf1 = shape_inference.infer_shapes(model)
print(f"Without annotation: Z = {get_shape(inf1.graph.output[0].type)}")

model.graph.value_info.append(
    helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["N", 256])
)
inf2 = shape_inference.infer_shapes(model)
print(f"With annotation:    Z = {get_shape(inf2.graph.output[0].type)}")
print("Annotating custom op output lets downstream MatMul infer [N, 10].")

In [ ]:
# Failure modes: when shape inference cannot complete
print("Shape Inference Failure Modes")
print("=" * 50)

# 1: Reshape with runtime target shape
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [2, 3, 4])
S = helper.make_tensor_value_info("S", TensorProto.INT64, [2])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
g = helper.make_graph([helper.make_node("Reshape", ["X", "S"], ["Y"])], "f1", [X, S], [Y])
m = helper.make_model(g, opset_imports=[helper.make_opsetid("", 17)])
inf = shape_inference.infer_shapes(m)
print(f"\n1. Reshape(runtime target): {get_shape(inf.graph.output[0].type)}")
print("   → Unknown because target shape is a runtime input.")

# 2: NonZero (data-dependent)
g2 = helper.make_graph(
    [helper.make_node("NonZero", ["X"], ["Y"])], "f2",
    [helper.make_tensor_value_info("X", TensorProto.FLOAT, [10, 20])],
    [helper.make_tensor_value_info("Y", TensorProto.INT64, None)],
)
m2 = helper.make_model(g2, opset_imports=[helper.make_opsetid("", 17)])
inf2 = shape_inference.infer_shapes(m2)
print(f"\n2. NonZero([10,20]): {get_shape(inf2.graph.output[0].type)}")
print("   → Second dim unknown (depends on how many elements ≠ 0).")

# 3: Custom op with no shape function
g3 = helper.make_graph(
    [helper.make_node("CustomAct", ["X"], ["Y"], domain="custom.ops")], "f3",
    [helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 256])],
    [helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)],
)
m3 = helper.make_model(g3, opset_imports=[
    helper.make_opsetid("", 17), helper.make_opsetid("custom.ops", 1)])
inf3 = shape_inference.infer_shapes(m3)
print(f"\n3. Custom op: {get_shape(inf3.graph.output[0].type)}")
print("   → No shape rule for 'custom.ops::CustomAct'.")

In [ ]:
# Manual shape inference implementations for common operators

def infer_matmul_shape(shape_a, shape_b):
    batch_a, batch_b = list(shape_a[:-2]), list(shape_b[:-2])
    batch_out = broadcast_shape(batch_a, batch_b) if (batch_a or batch_b) else []
    return batch_out + [shape_a[-2], shape_b[-1]]

def infer_conv2d_shape(n, c_out, h, w, k, p, s, d=1):
    return [n, c_out, (h+2*p-d*(k-1)-1)//s+1, (w+2*p-d*(k-1)-1)//s+1]

def infer_reshape_shape(shape_in, target):
    total = 1
    for d in shape_in: total *= d
    result, neg_idx, known = list(target), None, 1
    for i, d in enumerate(result):
        if d == -1: neg_idx = i
        else: known *= d
    if neg_idx is not None:
        result[neg_idx] = total // known
    return result

print("Manual shape inference:")
print("  MatMul [4,128] × [128,64] → ", infer_matmul_shape([4,128], [128,64]))
print("  MatMul [2,4,128] × [128,64] → ", infer_matmul_shape([2,4,128], [128,64]))
print("  Conv2D [1,3,32,32] k=3,p=1,s=1 → ", infer_conv2d_shape(1,64,32,32,3,1,1))
print("  Conv2D [1,3,224,224] k=7,p=3,s=2 → ", infer_conv2d_shape(1,64,224,224,7,3,2))
print("  Reshape [2,3,4] → [6,4] → ", infer_reshape_shape([2,3,4], [6,4]))
print("  Reshape [2,3,4] → [2,-1] → ", infer_reshape_shape([2,3,4], [2,-1]))
print("  Reshape [1,32,5,5] → [-1,800] → ", infer_reshape_shape([1,32,5,5], [-1,800]))

In [ ]:
# End-to-end: transformer block shape propagation
d_model, d_ff = 512, 2048

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["B", "S", d_model])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

inits = [
    numpy_helper.from_array(np.zeros((d_model, d_model), dtype=np.float32), n)
    for n in ["Wq", "Wk", "Wv", "Wo"]
] + [
    numpy_helper.from_array(np.zeros((d_model, d_ff), dtype=np.float32), "W1"),
    numpy_helper.from_array(np.zeros((d_ff, d_model), dtype=np.float32), "W2"),
]

nodes = [
    helper.make_node("MatMul", ["X", "Wq"], ["Q"]),
    helper.make_node("MatMul", ["X", "Wk"], ["K"]),
    helper.make_node("MatMul", ["X", "Wv"], ["V"]),
    helper.make_node("MatMul", ["Q", "Wo"], ["attn"]),
    helper.make_node("Add", ["X", "attn"], ["res1"]),
    helper.make_node("MatMul", ["res1", "W1"], ["ff_h"]),
    helper.make_node("Relu", ["ff_h"], ["ff_a"]),
    helper.make_node("MatMul", ["ff_a", "W2"], ["ff_o"]),
    helper.make_node("Add", ["res1", "ff_o"], ["Y"]),
]

graph = helper.make_graph(nodes, "transformer", [X], [Y], initializer=inits)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
inferred = shape_inference.infer_shapes(model)

print(f"Transformer block shape propagation ({len(nodes)} nodes):")
print(f"  Input X: ['B', 'S', {d_model}]")
for vi in inferred.graph.value_info:
    print(f"  {vi.name:5s}: {get_shape(vi.type)}")
print(f"  {'Y':5s}: {get_shape(inferred.graph.output[0].type)}  (output)")
print(f"\nSymbolic dims 'B','S' propagated; static: {d_model}→{d_model}, {d_model}→{d_ff}→{d_model}")

In [ ]:
# Visualize shape propagation through a CNN pipeline
import matplotlib.pyplot as plt

layers = [
    ("Input",  3,  224), ("Conv1", 64,  112), ("Pool1", 64,  56),
    ("Conv2", 128, 56),  ("Conv3", 256, 28),  ("Conv4", 512, 14),
    ("AvgPool",512, 1),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
names = [l[0] for l in layers]
chans = [l[1] for l in layers]
spatials = [l[2] for l in layers]

ax1.bar(range(len(names)), spatials, color="steelblue", edgecolor="black", alpha=0.85)
for i, h in enumerate(spatials):
    ax1.text(i, h + 3, str(h), ha="center", fontsize=9, fontweight="bold")
ax1.set_xticks(range(len(names)))
ax1.set_xticklabels(names, rotation=45, ha="right")
ax1.set_ylabel("Spatial Height")
ax1.set_title("Spatial Dimension Reduction", fontweight="bold")
ax1.grid(axis="y", alpha=0.3)

ax2.bar(range(len(names)), chans, color="coral", edgecolor="black", alpha=0.85)
for i, c in enumerate(chans):
    ax2.text(i, c + 5, str(c), ha="center", fontsize=9, fontweight="bold")
ax2.set_xticks(range(len(names)))
ax2.set_xticklabels(names, rotation=45, ha="right")
ax2.set_ylabel("Channels")
ax2.set_title("Channel Dimension Growth", fontweight="bold")
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("shape_propagation_cnn.png", dpi=100, bbox_inches="tight")
plt.show()
print("CNN pattern: spatial dims shrink while channel dims grow.")

## Key Takeaways

### Algorithm
- Shape inference is a **forward dataflow analysis** — $O(|V|+|E|)$ topological walk.
- $\text{shape}(\text{output}_i) = f_{\text{op}}(\text{shape}(\text{input}_1), \ldots, \text{shape}(\text{input}_k), \text{attrs})$
- Loops/subgraphs require **fixed-point iteration** until types stabilize.

### Per-Operator Rules

| Operator | Rule |
|:---------|:-----|
| MatMul | $[M, K] \otimes [K, N] \to [M, N]$, batch dims broadcast |
| Conv | $H_{out} = \lfloor(H_{in}+2p-d(k-1)-1)/s\rfloor + 1$ |
| Reshape | $\prod_i d_i = \prod_j d_j'$ (product conservation) |
| Concat | $d_{\text{axis}}^{\text{out}} = \sum_i d_{\text{axis}}^{\text{input}_i}$ |
| Broadcast | $d_i' = \max(d_i^A, d_i^B)$ when one is 1 |

### Practical Tips
- Use `infer_shapes(model, data_prop=True)` for maximum coverage.
- Run shape inference **before** `check_model()` for best error detection.
- Annotate `value_info` manually for custom ops.
- Expect **partial inference** for data-dependent ops (`NonZero`, `Where`).

---

**Next:** [Shape Inference — Apply](./Shape_Inference_Apply.ipynb) | [Model Validation — Deep Dive](../04_Model_Validation/Model_Validation_Deep_Dive.ipynb)